In [1]:
import os
import librosa
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder


In [19]:
import os
import librosa
from tqdm import tqdm

DATASET_PATH = "/content/drive/MyDrive/motor sound/IDMT-ISA-ELECTRIC-ENGINE/train_cut"

AUDIO_EXTENSIONS = (".wav", ".mp3")

min_duration = float("inf")
min_file = None

all_durations = []

for label_name in os.listdir(DATASET_PATH):
    label_path = os.path.join(DATASET_PATH, label_name)

    if not os.path.isdir(label_path):
        continue

    for file in tqdm(os.listdir(label_path), desc=f"Checking {label_name}"):
        if file.endswith(AUDIO_EXTENSIONS):
            file_path = os.path.join(label_path, file)

            try:
                y, sr = librosa.load(file_path, sr=None)  # original sample rate
                duration = len(y) / sr   # duration in seconds

                all_durations.append(duration)

                if duration < min_duration:
                    min_duration = duration
                    min_file = file_path

            except Exception as e:
                print(f"Error loading {file_path}: {e}")

print("\n✅ Minimum Duration:", min_duration, "seconds")
print("📂 File with minimum duration:", min_file)

Checking engine3_heavyload: 100%|██████████| 178/178 [00:02<00:00, 87.82it/s] 


✅ Minimum Duration: 3.0 seconds
📂 File with minimum duration: /content/drive/MyDrive/motor sound/IDMT-ISA-ELECTRIC-ENGINE/train_cut/engine2_broken/pure_21.wav


In [2]:
DATASET_PATH = "/content/drive/MyDrive/motor sound/IDMT-ISA-ELECTRIC-ENGINE/train_cut/engine1_good"  # train/good, train/overload, train/broken
SAMPLE_RATE = 1000
N_MFCC = 30
AUDIO_EXTENSIONS = (".wav", ".mp3")


In [3]:
file_paths = []
labels = []

for label_name in os.listdir("/content/drive/MyDrive/motor sound/IDMT-ISA-ELECTRIC-ENGINE/train_cut"):
    label_path = os.path.join("/content/drive/MyDrive/motor sound/IDMT-ISA-ELECTRIC-ENGINE/train_cut", label_name)

    if not os.path.isdir(label_path):
        continue

    for file in os.listdir(label_path):
        if file.endswith(AUDIO_EXTENSIONS):
            file_paths.append(os.path.join(label_path, file))
            labels.append(label_name)

print(f"✅ Total audio files found: {len(file_paths)}")


✅ Total audio files found: 507


In [ ]:
file_paths

['/content/drive/MyDrive/motor sound/IDMT-ISA-ELECTRIC-ENGINE/train_cut/engine2_broken/pure_21.wav',
 '/content/drive/MyDrive/motor sound/IDMT-ISA-ELECTRIC-ENGINE/train_cut/engine2_broken/pure_2.wav',
 '/content/drive/MyDrive/motor sound/IDMT-ISA-ELECTRIC-ENGINE/train_cut/engine2_broken/pure_13.wav',
 '/content/drive/MyDrive/motor sound/IDMT-ISA-ELECTRIC-ENGINE/train_cut/engine2_broken/pure_102.wav',
 '/content/drive/MyDrive/motor sound/IDMT-ISA-ELECTRIC-ENGINE/train_cut/engine2_broken/pure_109.wav',
 '/content/drive/MyDrive/motor sound/IDMT-ISA-ELECTRIC-ENGINE/train_cut/engine2_broken/pure_10.wav',
 '/content/drive/MyDrive/motor sound/IDMT-ISA-ELECTRIC-ENGINE/train_cut/engine2_broken/pure_20.wav',
 '/content/drive/MyDrive/motor sound/IDMT-ISA-ELECTRIC-ENGINE/train_cut/engine2_broken/pure_119.wav',
 '/content/drive/MyDrive/motor sound/IDMT-ISA-ELECTRIC-ENGINE/train_cut/engine2_broken/pure_118.wav',
 '/content/drive/MyDrive/motor sound/IDMT-ISA-ELECTRIC-ENGINE/train_cut/engine2_broken/p

In [4]:
labels

['engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_broken',
 'engine2_br

In [5]:
le = LabelEncoder()
encoded_labels = le.fit_transform(labels)

print("🔹 Label mapping:", dict(zip(le.classes_, le.transform(le.classes_))))


🔹 Label mapping: {np.str_('engine1_good'): np.int64(0), np.str_('engine2_broken'): np.int64(1), np.str_('engine3_heavyload'): np.int64(2)}


In [20]:
def extract_mfcc(file_path):
    y, sr = librosa.load(file_path, sr=SAMPLE_RATE,duration=3)
    if len(y) < SAMPLE_RATE * 1:
        y = np.pad(y, (0, SAMPLE_RATE * 1 - len(y)))
    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=N_MFCC
    )
    return mfcc


In [21]:
max_len = 0

for path in tqdm(file_paths, desc="Finding max MFCC length"):
    mfcc = extract_mfcc(path)
    max_len = max(max_len, mfcc.shape[1])

print(f"🔹 Max MFCC frame length: {max_len}")


Finding max MFCC length: 100%|██████████| 507/507 [00:09<00:00, 53.04it/s]

🔹 Max MFCC frame length: 6


In [22]:
def pad_or_trim(mfcc, max_len):
    if mfcc.shape[1] < max_len:
        pad_width = max_len - mfcc.shape[1]
        mfcc = np.pad(mfcc, ((0, 0), (0, pad_width)), mode="constant")
    else:
        mfcc = mfcc[:, :max_len]
    return mfcc


In [23]:
data_rows = []

for path, label in tqdm(zip(file_paths, encoded_labels), total=len(file_paths), desc="Extracting MFCC"):
    mfcc = extract_mfcc(path)
    mfcc = pad_or_trim(mfcc, max_len)

    features = mfcc.flatten()  # classical ML ready
    data_rows.append(np.append(features, label))


Extracting MFCC: 100%|██████████| 507/507 [00:10<00:00, 48.38it/s]


In [24]:
df_features = pd.DataFrame(data_rows)

feature_cols = [f"f{i}" for i in range(df_features.shape[1] - 1)]
df_features.columns = feature_cols + ["label"]

print("✅ Final dataset shape:", df_features.shape)


✅ Final dataset shape: (507, 181)


In [25]:
df_features['label'].replace(1.0, 'Broken', inplace=True)
df_features['label'].replace(0.0, 'Good', inplace=True)
df_features['label'].replace(2.0, 'Heavy Load', inplace=True)

/tmp/ipykernel_42322/1060815614.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_features['label'].replace(1.0, 'Broken', inplace=True)


In [26]:
df_features['label']

,label
0,Broken
1,Broken
2,Broken
3,Broken
4,Broken
...,...
502,Heavy Load
503,Heavy Load
504,Heavy Load
505,Heavy Load


In [27]:
OUTPUT_CSV = "motor_mfcc_embeddings.csv"
df_features.to_csv(OUTPUT_CSV, index=False)

print(f"💾 MFCC embeddings saved to: {OUTPUT_CSV}")


💾 MFCC embeddings saved to: motor_mfcc_embeddings.csv


#machine learning

In [29]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
import pandas as pd

# ===== Load Data =====
df = pd.read_csv("/content/motor_mfcc_embeddings.csv")

X = df.drop(columns=["label"])
y = df["label"]

# ===== Encode Labels =====
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# ===== Train Test Split =====
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42
)

# ===== Pipeline =====
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", XGBClassifier(
        eval_metric='logloss'
    ))
])

# ===== Train =====
pipeline.fit(X_train, y_train)

# ===== Prediction =====
y_pred = pipeline.predict(X_test)

# ===== Evaluation =====
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nReport:\n", classification_report(y_test, y_pred))

Accuracy: 0.9215686274509803

Report:
               precision    recall  f1-score   support

           0       0.89      0.92      0.91        37
           1       0.85      0.92      0.88        24
           2       1.00      0.93      0.96        41

    accuracy                           0.92       102
   macro avg       0.91      0.92      0.92       102
weighted avg       0.93      0.92      0.92       102



In [30]:
import pickle

# Save the trained pipeline to a .pkl file
with open('model.pkl', 'wb') as file:
    pickle.dump(pipeline, file)

print("✅ Model saved to model.pkl")

✅ Model saved to model.pkl


In [31]:
import pickle

# Save the LabelEncoder to a .pkl file
with open('label_encoder.pkl', 'wb') as file:
    pickle.dump(le, file)

print("✅ LabelEncoder saved to label_encoder.pkl")

✅ LabelEncoder saved to label_encoder.pkl


infrence

In [32]:
import pickle
import librosa
import numpy as np

# Load trained pipeline
with open('model.pkl', 'rb') as f:
    pipeline = pickle.load(f)

# Load label encoder (IMPORTANT)
with open('label_encoder.pkl', 'rb') as f:
    le = pickle.load(f)

In [33]:
SAMPLE_RATE = 1000
N_MFCC = 30
DURATION = 3      # 🔥 updated
MAX_LEN = 6       # 🔥 updated

In [34]:
def extract_mfcc(file_path):
    y, sr = librosa.load(file_path, sr=SAMPLE_RATE, duration=DURATION)

    # pad if shorter than 3 sec
    expected_len = SAMPLE_RATE * DURATION
    if len(y) < expected_len:
        y = np.pad(y, (0, expected_len - len(y)))

    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=N_MFCC,
        n_fft=512   # 🔥 important fix
    )

    return mfcc


def pad_or_trim(mfcc, max_len):
    if mfcc.shape[1] < max_len:
        pad_width = max_len - mfcc.shape[1]
        mfcc = np.pad(mfcc, ((0, 0), (0, pad_width)), mode="constant")
    else:
        mfcc = mfcc[:, :max_len]

    return mfcc

In [35]:
def predict_audio(file_path):

    # 1️⃣ Extract MFCC
    mfcc = extract_mfcc(file_path)

    # 2️⃣ Pad / Trim
    mfcc = pad_or_trim(mfcc, MAX_LEN)

    # 3️⃣ Flatten
    features = mfcc.flatten().reshape(1, -1)

    # 4️⃣ Predict
    pred = pipeline.predict(features)

    return pred[0]

In [36]:
def predict_label(file_path):
    pred = predict_audio(file_path)
    label = le.inverse_transform([pred])[0]
    return label

In [37]:
def predict_with_confidence(file_path):
    mfcc = extract_mfcc(file_path)
    mfcc = pad_or_trim(mfcc, MAX_LEN)
    features = mfcc.flatten().reshape(1, -1)

    probs = pipeline.predict_proba(features)
    pred = np.argmax(probs)

    label = le.inverse_transform([pred])[0]
    confidence = np.max(probs)

    return label, confidence

In [41]:
audio_path = "/content/drive/MyDrive/motor sound/IDMT-ISA-ELECTRIC-ENGINE/test/engine3_heavyload/atmo_low.wav"

result = predict_with_confidence(audio_path)

print("🎯 Prediction:", result)

🎯 Prediction: ('Heavy Load', np.float32(0.9951356))


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
